# Mitochondrial translation reporter — flow cytometry

Analysis of Cox2-sfGFP reporter expression across PDH complex mutants.

## Reporter

Mitochondrial translation is monitored using a mitochondrially encoded
superfolder GFP reporter (*sfGFP*ᵐ) integrated at the *cox2* locus, as described
in Suhm et al. (2018). Because the reporter is encoded on mtDNA, GFP
fluorescence requires functional mitochondrial translation — so signal loss
reports directly on translation rather than on nuclear gene expression or
protein import.

## Growth and treatment

Cells were grown overnight at 30 °C in YPD, then back-diluted the following
morning into YPG. For the temperature-shift condition, cultures were grown for
11 h at either 30 °C or 37 °C. For chloramphenicol treatment, cultures were
grown at 30 °C and treated with 1 mg/ml CAP for 5 h. Cultures were maintained in
mid-log phase until sampling.

CAP inhibits mitochondrial translation specifically, and serves as the
specificity control for the reporter.

## Acquisition and gating

Samples were analysed on a BD Accuri C6 Plus. Initial gating was performed in
**floreada.io**; this notebook reads the exported per-event CSVs, one file per
sample, and performs all subsequent analysis.

Cells were gated on forward-scatter area versus height (FSC-A vs FSC-H) to
exclude non-cell events such as debris and agar particles.


## Strains

| Genotype | Description |
|---|---|
| WT | Reference |
| Δpda1 | PDH E1α deletion |
| Δpda1+pda1wt | Δpda1 complemented with wild-type *PDA1* |
| Δpda1+pda1G | Δpda1 complemented with catalytically inactive variant |
| Δlat1 | PDH E2 deletion |
| Δlat1+lat1wt | Δlat1 complemented with wild-type *LAT1* |
| Δlat1+lat1HD | Δlat1 complemented with catalytically inactive variant |

The complemented strains are the key controls: if the wild-type construct
restores reporter expression and the catalytically inactive variant does not,
the phenotype depends on PDH catalytic activity rather than on the protein's
presence.

## Analysis conventions

- **Measurements:** FITC-A normalised to FSC-A per event, summarised as the median per sample. Normalising to forward scatter controls for cell size, which differs between conditions; note this is a proxy for cell size, not for mitochondrial mass.
- **Unit of replication:** the biological replicate (n = 3), not the event. Flow data give tens of thousands of events per sample, and testing on pooled events would reflect event count rather than biological variation. Figures are SuperPlots: individual events shown faintly, replicate medians as large points, statistics computed on the replicate medians.
- **Relative expression:** each replicate's mutant medians are divided by that same replicate's WT median at the same temperature, preserving pairing and removing between-run differences in absolute fluorescence.
- **Tests:** mutant vs WT by one-sample *t*-test against μ = 1 (WT divided by itself has zero variance, so a paired test is degenerate); mutant vs mutant by paired *t*-test on relative values.
- **Comparisons shown** are a pre-specified set of six: each deletion vs WT, and each deletion vs its two complemented strains.
- **Display capping:** event-level plots are capped at the 99.5th percentile for legibility; all statistics use the full data.

## Running this notebook

Set `DATA_DIR` (gated CSV exports) and `OUT_DIR` (figures) in the loading cell.

**Expected filename pattern:** `events-<well> <genotype>_<condition>_<replicate>.fcs.csv`.
Metadata is parsed by position after splitting on underscores, so files must
follow this convention.

**Requires:** `pandas`, `numpy`, `scipy`, `seaborn`, `matplotlib`


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



## 1. Setup and data loading

Loads per-event CSV exports from floreada.io — one file per sample, containing
events that passed the FSC-A vs FSC-H gate — and assembles them into a single
event-level table.

**Filename parsing.** The well prefix written by the export (everything before
the first space) and the `.fcs.csv` extension are stripped, and the remainder is
split on underscores into genotype, condition, and replicate, which are attached
as columns. Expected pattern: `events-<well> <genotype>_<condition>_<replicate>.fcs.csv`.

**Derived measures:**

- `log_FITC` — log10 of the FITC signal. Flow cytometry fluorescence is
  approximately log-normally distributed, so the log scale is the appropriate
  one for both visualisation and summary statistics. A pseudocount of 1 avoids
  log(0) for any events at baseline.
- `FITC_per_Size` — FITC divided by FSC-A, normalising reporter signal to cell
  size. Larger cells contain more mitochondria and therefore more reporter, so
  this separates changes in per-cell reporter expression from changes in cell
  size — relevant here since temperature shift and CAP treatment both affect
  cell size.

In [ ]:

DATA_DIR = '.'
data_dir = f'{DATA_DIR}/all_data_3/'

file_pattern = os.path.join(data_dir, '*.csv')
files = glob.glob(file_pattern)

df_list = []

for file in files:
    basename = os.path.basename(file)
    
    # 1. Strip the well prefix (everything before the first space)
    # e.g., "events-A01 615_Gal_1_2h.fcs.csv" -> "615_Gal_1_2h.fcs.csv"
    if ' ' in basename:
        clean_name = basename.split(' ', 1)[1]
    else:
        clean_name = basename
        
    # 2. Remove extensions
    clean_name = clean_name.replace('.fcs.csv', '')
    
    # 3. Extract metadata: Genotype, Condition, Replicate, Timepoint
    # Splitting "615_Gal_1_2h" by underscores
    parts = clean_name.split('_')
    
    genotype = parts[0] if len(parts) > 0 else 'unknown'
    condition = parts[1] if len(parts) > 1 else 'unknown'
    replicate = parts[2] if len(parts) > 2 else 'unknown'
    #timepoint = parts[3] if len(parts) > 3 else 'unknown'
    
    # Read the data
    df = pd.read_csv(file)
    
    # Append metadata columns to the dataframe
    df['Genotype'] = genotype
    df['Condition'] = condition
    df['Replicate'] = replicate
    #df['Timepoint'] = timepoint
    
    df_list.append(df)

# Concatenate all events into a single Master DataFrame
master_df = pd.concat(df_list, ignore_index=True)
master_df

In [ ]:
# --- Configure Parameters ---
# IMPORTANT: Replace these strings with the exact column headers from your CSV
fsc_col = 'FSC-A'   
fitc_col = 'FITC-A::FL1-A' 

# Set a clean, publication-ready style
sns.set_theme(style="ticks", context="paper", font_scale=1.2)

# Flow cytometry fluorescence is typically log-normally distributed.
# Log10 transformation is highly recommended for accurate FITC visualization.
# We use +1 to avoid log(0) errors if any baseline events hit 0.
master_df['log_FITC'] = np.log10(master_df[fitc_col] + 1)
master_df['FITC_per_Size'] = master_df[fitc_col] / master_df[fsc_col]

## 2. Validate metadata and aggregate to replicate level

**Validation.** Prints the parsed genotypes, conditions, and replicates, plus
event counts per group, so that any filename that parsed incorrectly is visible
before the analysis proceeds.

**Genotype labels.** Filename tokens are mapped to readable strain names and set
as an ordered categorical, so every figure follows the same left-to-right
sequence. Unmapped values fall through to the raw token rather than becoming
`NaN`, so a missing entry in the map shows up as an odd label rather than
disappearing.

**Aggregation.** Events are collapsed to one row per genotype × condition ×
replicate:

| Column | Meaning |
|---|---|
| `n_events` | Gated events contributing to that replicate |
| `median_FITC` | Median log10 FITC — the primary summary |
| `mean_FITC` | Mean log10 FITC |
| `cv_FITC` | SD/mean of log10 FITC as a percentage |
| `median_norm` | Median FITC normalised to FSC-A |

Aggregating to replicate level before testing is the point of this step: flow
data give tens of thousands of events per sample, but the unit of replication is
the biological replicate, not the event. Statistics computed on pooled events
would reflect event count rather than biological variation.

The median is used as the primary summary because fluorescence distributions are
log-normal and often skewed, so the median is more robust than the mean to the
tail of bright events.

In [ ]:
# ── Cell 1: Validate & clean metadata ──────────────────────────────────────
print("Shape:", master_df.shape)
print("\nGenotypes found:", master_df['Genotype'].unique())
print("Conditions found:", master_df['Condition'].unique())
print("Replicates found:", master_df['Replicate'].unique())
print("\nEvents per group:")
print(master_df.groupby(['Genotype', 'Condition']).size().unstack(fill_value=0))

# ── Harmonise genotype names to readable labels ───────────────────────────
# Adjust this dict to match whatever your file names actually produced
genotype_map = {
    'wt'   : 'WT',
    'dpdb' : '∆pdb1',
    'dpda' : 'Δpda1',
    'dlat' : 'Δlat1',
    'dlat+lat' : '∆lat1+lat1wt',
    'dlat+latHD' : '∆lat1+lat1HD',
    'dpda+pda' : '∆pda1+pda1wt',
    'dpda+pdaG' : '∆pda1+pda1G',
    
    
}
master_df['Genotype_label'] = master_df['Genotype'].map(genotype_map).fillna(master_df['Genotype'])

# Ordered categories so plots always follow the same left-to-right sequence
GENOTYPE_ORDER = [ 'WT', 'Δpda1', '∆pdb1', 'Δlat1', '∆pda1+pda1wt', '∆pda1+pda1G', '∆lat1+lat1wt', '∆lat1+lat1HD' ]
TEMP_ORDER     = ['30', '37']          # adjust if condition column stores integers
PALETTE        = {'30': '#4C9BE8', '37': '#E8714C'}   # blue = permissive, orange = stress

master_df['Genotype_label'] = pd.Categorical(master_df['Genotype_label'],
                                              categories=GENOTYPE_ORDER, ordered=True)
master_df['Condition']      = master_df['Condition'].astype(str)   # ensure string for hue
print("\nAfter remapping:")
print(master_df.groupby(['Genotype_label', 'Condition']).size().unstack(fill_value=0))

In [ ]:
# ── Cell 3: Aggregate to replicate-level medians ──────────────────────────
rep_stats = (
    master_df
    .groupby(['Genotype_label', 'Condition', 'Replicate'])
    .agg(
        n_events      = ('log_FITC', 'count'),
        median_FITC   = ('log_FITC', 'median'),
        mean_FITC     = ('log_FITC', 'mean'),
        cv_FITC       = ('log_FITC', lambda x: x.std() / x.mean() * 100),  # %CV
        median_norm   = ('FITC_per_Size', 'median'),   # size-normalised
    )
    .reset_index()
)

rep_stats['Genotype_label'] = pd.Categorical(rep_stats['Genotype_label'],
                                              categories=GENOTYPE_ORDER, ordered=True)
rep_stats = rep_stats.sort_values(['Genotype_label', 'Condition', 'Replicate'])
print(rep_stats.to_string(index=False))
rep_stats.to_csv('rep_level_medians.csv', index=False)

In [ ]:
# ── Cell 4: Bar + individual replicate dots ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, temp in zip(axes, TEMP_ORDER):
    sub = rep_stats[rep_stats['Condition'] == temp]

    # Bar = mean across replicates; error bar = SEM
    grp = (sub.groupby('Genotype_label', observed=True)['median_FITC']
              .agg(['mean', 'sem', 'count'])
              .reindex(GENOTYPE_ORDER)
              .reset_index())

    ax.bar(
        grp['Genotype_label'], grp['mean'],
        yerr=grp['sem'], capsize=4,
        color=PALETTE[temp], alpha=0.6,
        error_kw=dict(elinewidth=1, ecolor='black')
    )

    # Overlay individual replicate dots (jittered x)
    import random
    random.seed(42)
    for _, row in sub.iterrows():
        xi = GENOTYPE_ORDER.index(row['Genotype_label'])
        jitter = (random.random() - 0.5) * 0.25
        ax.plot(xi + jitter, row['median_FITC'],
                'o', color='black', ms=5, zorder=5, alpha=0.85)

    ax.set_xticks(range(len(GENOTYPE_ORDER)))
    ax.set_xticklabels(GENOTYPE_ORDER, rotation=25, ha='right')
    ax.set_title(f'{temp} °C', fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('Median log₁₀(FITC + 1)' if ax == axes[0] else '')
    sns.despine(ax=ax)

fig.suptitle('Cox2-sfGFP expression — replicate medians ± SEM', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Paired Wilcoxon signed-rank test (replicates matched across genotypes) ─
# Justified because: all genotypes measured same day → replicate 1 of WT is
# the paired control for replicate 1 of Δpda1, etc.

from scipy.stats import ttest_rel

def pval_to_stars(p):
    if p < 0.001: return '***'
    elif p < 0.01:  return '**'
    elif p < 0.05:  return '*'
    else:           return 'ns'

def compute_paired_pvals(rep_stats, temp, value_col='median_norm'):
    """
    Returns a dict {(g1, g2): p_value} using Wilcoxon signed-rank on
    matched replicate medians.
    """
    sub = rep_stats[rep_stats['Condition'] == temp].copy()
    sub['Replicate'] = sub['Replicate'].astype(str)
    
    # Pivot: rows = replicates, columns = genotypes
    pivot = (sub.pivot_table(index='Replicate',
                             columns='Genotype_label',
                             values=value_col))
    pivot = pivot.reindex(columns=GENOTYPE_ORDER)
    
    print(f"\n── {temp} °C  — replicate-matched medians (FITC/FSC) ──")
    print(pivot.to_string())
    
    pval_dict = {}
    for g1, g2 in combinations(GENOTYPE_ORDER, 2):
        if g1 not in pivot.columns or g2 not in pivot.columns:
            continue
        paired = pivot[[g1, g2]].dropna()
        d = paired[g1].values - paired[g2].values
        
        if len(paired) < 3:
            print(f"  {g1} vs {g2}: not enough paired obs (n={len(paired)})")
            pval_dict[(g1, g2)] = 1.0
            continue
        
        if np.all(d == 0):
            p = 1.0
        else:
            try:
                # zero_method='wilcox' drops zero differences (standard)
                _, p = ttest_rel(paired[g1].values, paired[g2].values)
            except ValueError:
                p = 1.0
        
        direction = '↑' if paired[g1].mean() > paired[g2].mean() else '↓'
        print(f"  {g1} vs {g2}: p={p:.4f} {pval_to_stars(p)}  ({g1} {direction} vs {g2})")
        pval_dict[(g1, g2)] = p
    
    return pval_dict

# Compute for both temperatures
pvals_30 = compute_paired_pvals(rep_stats, '30')
pvals_37 = compute_paired_pvals(rep_stats, '37')
pval_lookup = {'30': pvals_30, '37': pvals_37}

## 3. SuperPlots — Cox2-sfGFP expression across PDH mutants

Three related outputs: absolute size-normalised expression, expression relative
to WT, and a summary heatmap.

**Size-normalised (first panel pair).** Replicate medians of FITC/FSC-A, plotted
per genotype at 30 °C and 37 °C. Each panel shows individual events faintly in
the background (coloured by replicate, subsampled), replicate medians as large
points, and the grand mean as a horizontal bar. The display is capped at the
99.5th percentile so a small number of very bright events do not compress the
axis; the statistics use the full data.

**Relative to WT (main figure).** Each replicate's mutant medians are divided by
that same replicate's WT median, within the same temperature. This preserves
pairing — rep 1 mutants against rep 1 WT — and removes between-replicate
differences in absolute fluorescence, which vary with instrument gain and
culture state.

**Statistical tests.** Two different tests are used, by necessity:

- **Mutant vs WT** — one-sample *t*-test against μ = 1. WT divided by itself is
  exactly 1 with zero variance, so a paired test is degenerate.
- **Mutant vs mutant** — paired *t*-test on relative values.

Comparisons shown are restricted to the biologically meaningful set: each
deletion against WT, and each deletion against its own complemented strains
(wild-type and catalytically-inactive variants).

**Heatmap.** Mean relative expression per genotype × temperature, colour-scaled
around 1.0 (WT level), with WT cells marked as the reference.

In [ ]:
# ── Cell A: shared setup ───────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import seaborn as sns
import pandas as pd
from scipy.stats import ttest_rel
from itertools import combinations

sns.set_theme(style="ticks", context="paper", font_scale=1.2)
plt.rcParams.update({
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         False,
})

GENOTYPE_ORDER = [ 'WT', 'Δpda1', '∆pda1+pda1wt', '∆pda1+pda1G',   'Δlat1', '∆lat1+lat1wt', '∆lat1+lat1HD' ]
TEMP_ORDER     = ['30', '37']  
REP_COLORS     = ["#BD5B28", "#4d6171", "#88A27D"]   # purple, olive, cyan — matches reference

def pval_to_stars(p):
    if p < 0.001:  return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    elif p < 0.1:  return '†'
    else:          return 'ns'

def compute_paired_pvals(rep_stats_df, temp, value_col):
    """Paired two-tailed t-test on matched replicate medians."""
    sub = rep_stats_df[rep_stats_df['Condition'] == temp].copy()
    sub['Replicate'] = sub['Replicate'].astype(str)
    pivot = (sub.pivot_table(index='Replicate',
                             columns='Genotype_label',
                             values=value_col)
               .reindex(columns=GENOTYPE_ORDER))
    print(f"\n── {temp} °C  [{value_col}] ──")
    print(pivot.round(6).to_string())
    pval_dict = {}
    for g1, g2 in combinations(GENOTYPE_ORDER, 2):
        if g1 not in pivot.columns or g2 not in pivot.columns:
            continue
        paired = pivot[[g1, g2]].dropna()
        if len(paired) < 3:
            pval_dict[(g1, g2)] = 1.0
            continue
        t_stat, p = ttest_rel(paired[g1].values, paired[g2].values)
        diff = paired[g1].values - paired[g2].values
        dz   = diff.mean() / diff.std(ddof=1)
        direction = '↑' if paired[g1].mean() > paired[g2].mean() else '↓'
        print(f"  {g1} vs {g2}: t={t_stat:.3f}, p={p:.4f} "
              f"{pval_to_stars(p)}  dz={dz:.2f}  ({g1}{direction}{g2})")
        pval_dict[(g1, g2)] = p
    return pval_dict

def annotate_brackets(ax, x1, x2, y, p, h, lw=0.9):
    """Bracket + star annotation between two x positions."""
    stars = pval_to_stars(p)
    color = '#222222' if stars != 'ns' else '#aaaaaa'
    lw_use = 1.0 if stars != 'ns' else 0.6
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y],
            lw=lw_use, c=color, clip_on=False, solid_capstyle='butt')
    ax.text((x1+x2)/2, y + h*1.1, stars,
            ha='center', va='bottom',
            fontsize=10 if stars != 'ns' else 8.5,
            color=color,
            fontweight='bold' if stars not in ('ns', '†') else 'normal')

def make_superplot(ax, temp, y_col, ylabel,
                   plot_df_use, rep_stats_use, rep_val_col, pval_lookup,
                   allowed_pairs=None):

    sub_all = plot_df_use[plot_df_use['Condition'] == temp].copy()
    sub_rep = rep_stats_use[rep_stats_use['Condition'] == temp].copy()
    sub_all['Replicate'] = sub_all['Replicate'].astype(str)
    sub_rep['Replicate'] = sub_rep['Replicate'].astype(str)

    reps        = sorted(sub_all['Replicate'].unique())
    rep_palette = {r: REP_COLORS[i % len(REP_COLORS)] for i, r in enumerate(reps)}

    # ── 1. Background single-cell stripplot (faint, per-replicate colour) ──
    for gi, geno in enumerate(GENOTYPE_ORDER):
        g_data = sub_all[sub_all['Genotype_label'] == geno]
        for rep in reps:
            r_data = g_data[g_data['Replicate'] == rep][y_col]
            n      = min(len(r_data), 800)
            r_samp = r_data.sample(n=n, random_state=42) if len(r_data) > n else r_data
            np.random.seed(int(rep) if rep.isdigit() else 42)
            jitter = np.random.uniform(-0.25, 0.25, size=len(r_samp))
            ax.scatter(
                gi + jitter, r_samp,
                color=rep_palette[rep],
                alpha=0.12, s=4, linewidths=0,
                rasterized=True, zorder=1,
            )

    # ── 2. Mean line (horizontal bar at the grand mean across replicates) ──
    for gi, geno in enumerate(GENOTYPE_ORDER):
        vals = sub_rep[sub_rep['Genotype_label'] == geno][rep_val_col]
        mn   = vals.mean()
        ax.plot([gi - 0.22, gi + 0.22], [mn, mn],
                color='#444444', lw=2.0, solid_capstyle='round', zorder=4)

    # ── 3. Swarmplot of replicate medians (large bold dots) ────────────────
    swarm_df = sub_rep[['Genotype_label', 'Replicate', rep_val_col]].copy()
    swarm_df['Genotype_label'] = pd.Categorical(
        swarm_df['Genotype_label'], categories=GENOTYPE_ORDER, ordered=True)

    sns.swarmplot(
        data      = swarm_df,
        x         = 'Genotype_label',
        y         = rep_val_col,
        hue       = 'Replicate',
        hue_order = reps,
        palette   = rep_palette,
        order     = GENOTYPE_ORDER,
        size      = 11,
        linewidth = 0.6,
        edgecolor = 'white',
        zorder    = 5,
        ax        = ax,
        legend    = False,
    )

    # ── 4. Paired t-test brackets ──────────────────────────────────────────
    pvals    = pval_lookup.get(temp, {})
    y_top    = sub_all[y_col].quantile(0.995)

    bracket_step = y_top * 0.09
    bracket_h    = y_top * 0.022
    y_base       = y_top * 1.05

    pair_list = sorted(combinations(range(len(GENOTYPE_ORDER)), 2),
                       key=lambda x: x[1] - x[0])
    if allowed_pairs is not None:
        pair_list = [pr for pr in pair_list if frozenset(pr) in allowed_pairs]

    for tier, (i, j) in enumerate(pair_list):
        g1, g2 = GENOTYPE_ORDER[i], GENOTYPE_ORDER[j]
        p = pvals.get((g1, g2), pvals.get((g2, g1), 1.0))
        annotate_brackets(ax, i, j,
                          y_base + tier * bracket_step,
                          p, h=bracket_h)

    ax.set_ylim(0, y_base + max(len(pair_list), 1) * bracket_step + bracket_h * 4)

    # ── 5. Aesthetics ──────────────────────────────────────────────────────
    ax.set_xticks(range(len(GENOTYPE_ORDER)))
    ax.set_xticklabels(GENOTYPE_ORDER, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_xlabel('')
    ax.set_title(f'{temp} °C  |  YPG 11 h', fontsize=11,
                 fontweight='bold', pad=8)
    ax.set_xlim(-0.6, len(GENOTYPE_ORDER) - 0.4)
    sns.despine(ax=ax, offset=6, trim=False)

    # ── 6. Legend ──────────────────────────────────────────────────────────
    dot_handles = [
        mlines.Line2D([0], [0], marker='o', color='w',
                      markerfacecolor=rep_palette[r],
                      markeredgecolor='white',
                      markersize=9, label=r)
        for r in reps
    ]
    ax.legend(handles=dot_handles,
              title='', frameon=True, framealpha=0.9,
              edgecolor='#cccccc',
              fontsize=9, loc='upper right',
              handletextpad=0.3, borderpad=0.6)

print("Setup complete.")

In [ ]:
# ── Cell B: size-normalised ────────────────────────────────────────────────

rep_stats_norm = (
    master_df
    .groupby(['Genotype_label', 'Condition', 'Replicate'])
    .agg(median_norm=('FITC_per_Size', 'median'))
    .reset_index()
)
rep_stats_norm['Genotype_label'] = pd.Categorical(
    rep_stats_norm['Genotype_label'], categories=GENOTYPE_ORDER, ordered=True)

pvals_norm_30    = compute_paired_pvals(rep_stats_norm, '30', 'median_norm')
pvals_norm_37    = compute_paired_pvals(rep_stats_norm, '37', 'median_norm')
pval_lookup_norm = {'30': pvals_norm_30, '37': pvals_norm_37}

q995         = master_df['FITC_per_Size'].quantile(0.995)
plot_df_norm = master_df[master_df['FITC_per_Size'] <= q995].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=False)
for ax, temp in zip(axes, TEMP_ORDER):
    make_superplot(
        ax            = ax,
        temp          = temp,
        y_col         = 'FITC_per_Size',
        ylabel        = 'FITC-A / FSC-A',
        plot_df_use   = plot_df_norm,
        rep_stats_use = rep_stats_norm,
        rep_val_col   = 'median_norm',
        pval_lookup   = pval_lookup_norm,
    )

fig.suptitle('Cox2-sfGFP expression (FITC / FSC-A, size-normalised)\nPDH complex mutants — YPD → YPG shift',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout(w_pad=4)

plt.show()
print("Saved.")

In [ ]:
# ── Cell B: relative expression (normalised to paired WT within replicate) ─

# Step 1: get replicate-level medians for FITC/FSC (absolute)
rep_stats_norm = (
    master_df
    .groupby(['Genotype_label', 'Condition', 'Replicate'])
    .agg(median_norm=('FITC_per_Size', 'median'))
    .reset_index()
)
rep_stats_norm['Genotype_label'] = pd.Categorical(
    rep_stats_norm['Genotype_label'], categories=GENOTYPE_ORDER, ordered=True)
rep_stats_norm['Replicate'] = rep_stats_norm['Replicate'].astype(str)

# Step 2: for each replicate × temperature, divide by that replicate's WT median
# This preserves pairing: rep1 mutants / rep1 WT, rep2 mutants / rep2 WT etc.
wt_ref = (rep_stats_norm[rep_stats_norm['Genotype_label'] == 'WT']
          .set_index(['Condition', 'Replicate'])['median_norm']
          .rename('wt_median'))

rep_stats_rel = rep_stats_norm.join(
    wt_ref, on=['Condition', 'Replicate']
)
rep_stats_rel['rel_expression'] = (rep_stats_rel['median_norm']
                                   / rep_stats_rel['wt_median'])
rep_stats_rel['Genotype_label'] = pd.Categorical(
    rep_stats_rel['Genotype_label'], categories=GENOTYPE_ORDER, ordered=True)

print("Relative expression per replicate:")
print(rep_stats_rel[['Genotype_label', 'Condition', 'Replicate',
                      'median_norm', 'wt_median', 'rel_expression']]
      .sort_values(['Condition', 'Genotype_label', 'Replicate'])
      .round(4).to_string(index=False))

# Step 3: paired t-test on relative expression values
# WT replicates will be ~1.0 (not exactly 1.0 — each is its own/own,
# so WT rep1 / WT rep1 = 1 exactly. To avoid this degeneracy for the
# WT vs X comparisons, we test mutants vs the value 1.0 using one-sample t-test,
# and mutant vs mutant using paired t-test as before.
from scipy.stats import ttest_1samp

def compute_relative_pvals(rep_stats_rel_df, temp):
    """
    WT vs mutant: one-sample t-test against mu=1.0 (WT is reference=1 by definition).
    Mutant vs mutant: paired t-test on relative expression values.
    """
    sub  = rep_stats_rel_df[rep_stats_rel_df['Condition'] == temp].copy()
    pivot = (sub.pivot_table(index='Replicate',
                             columns='Genotype_label',
                             values='rel_expression')
               .reindex(columns=GENOTYPE_ORDER))

    print(f"\n── {temp} °C  [relative expression] ──")
    print(pivot.round(4).to_string())

    pval_dict = {}
    for g1, g2 in combinations(GENOTYPE_ORDER, 2):
        if g1 not in pivot.columns or g2 not in pivot.columns:
            continue
        v1 = pivot[g1].dropna().values
        v2 = pivot[g2].dropna().values

        if g1 == 'WT':
            # one-sample: are mutant values different from 1?
            t_stat, p = ttest_1samp(v2, popmean=1.0)
            test_used = 'one-sample vs 1'
        elif g2 == 'WT':
            t_stat, p = ttest_1samp(v1, popmean=1.0)
            test_used = 'one-sample vs 1'
        else:
            # paired t-test between two mutants
            paired = pivot[[g1, g2]].dropna()
            t_stat, p = ttest_rel(paired[g1].values, paired[g2].values)
            test_used = 'paired t-test'

        direction = '↑' if v1.mean() > v2.mean() else '↓'
        print(f"  {g1} vs {g2}: t={t_stat:.3f}, p={p:.4f} "
              f"{pval_to_stars(p)}  [{test_used}]  ({g1}{direction}{g2})")
        pval_dict[(g1, g2)] = p
    return pval_dict

pvals_rel_30   = compute_relative_pvals(rep_stats_rel, '30')
pvals_rel_37   = compute_relative_pvals(rep_stats_rel, '37')
pval_lookup_rel = {'30': pvals_rel_30, '37': pvals_rel_37}

In [ ]:
# ── Cell C: main figure superplot — relative expression ───────────────────

# Single-cell relative values: divide each cell's FITC/FSC by its
# replicate's WT median (same reference used for the stats)
plot_df_rel = master_df.copy()
plot_df_rel['Replicate'] = plot_df_rel['Replicate'].astype(str)

plot_df_rel = plot_df_rel.merge(
    wt_ref.reset_index(),
    on=['Condition', 'Replicate'],
    how='left'
)
plot_df_rel['rel_FITC'] = plot_df_rel['FITC_per_Size'] / plot_df_rel['wt_median']

# Cap outliers on relative scale
q995_rel    = plot_df_rel['rel_FITC'].quantile(0.995)
plot_df_rel = plot_df_rel[plot_df_rel['rel_FITC'] <= q995_rel].copy()

# WT=0, Δpda1=1, ∆pda1+pda1wt=2, ∆pda1+pda1G=3,
# Δlat1=4, ∆lat1+lat1wt=5, ∆lat1+lat1HD=6
ALLOWED_PAIRS = {
    frozenset({0, 1}),   # WT       vs Δpda1
    frozenset({0, 4}),   # WT       vs Δlat1
    frozenset({1, 2}),   # Δpda1    vs ∆pda1+pda1wt
    frozenset({1, 3}),   # Δpda1    vs ∆pda1+pda1G
    frozenset({4, 5}),   # Δlat1    vs ∆lat1+lat1wt
    frozenset({4, 6}),   # Δlat1    vs ∆lat1+lat1HD
}

# ── Draw ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=False)

for ax, temp in zip(axes, TEMP_ORDER):
    make_superplot(
        ax            = ax,
        temp          = temp,
        y_col         = 'rel_FITC',
        ylabel        = 'Relative Cox2-sfGFP expression\n(normalised to WT)',
        plot_df_use   = plot_df_rel,
        rep_stats_use = rep_stats_rel,
        rep_val_col   = 'rel_expression',
        pval_lookup   = pval_lookup_rel,
        allowed_pairs = ALLOWED_PAIRS
    )
    # Add reference line at y=1 (WT level)
    ax.axhline(1.0, color='#999999', lw=0.8, ls='--', zorder=0)
    ax.text(len(GENOTYPE_ORDER) - 0.38, 1.0, 'WT ref',
            va='bottom', ha='right', fontsize=7.5,
            color='#999999', style='italic')

fig.suptitle('Cox2-sfGFP expression relative to WT\nPDH complex mutants — YPD → YPG shift, 11 h',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout(w_pad=4)
plt.rcParams['svg.fonttype'] = 'none'
plt.savefig(f'{OUT_DIR}/plot_relative_090626_main.svg')
plt.show()
print("Saved main figure.")

In [ ]:
# ── Heatmap: relative Cox2-sfGFP expression (normalised to WT) ────────────

pivot_rel = (rep_stats_rel
             .groupby(['Genotype_label', 'Condition'], observed=True)['rel_expression']
             .mean()
             .unstack('Condition')
             .reindex(GENOTYPE_ORDER))

# Optional: reorder columns to 30 → 37
pivot_rel = pivot_rel.reindex(columns=['30', '37'])

fig, ax = plt.subplots(figsize=(4, 4))
sns.heatmap(
    pivot_rel,
    annot=True, fmt='.2f',
    cmap=sns.color_palette(["#b6c9c1", "#88A27D", "#4d6171", "#162734"], as_cmap=True),
    vmin=0.6, vmax=1.4,          # centre the colorbar around 1 (WT level)
    linewidths=0.5,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Expression relative to WT',
              'shrink': 0.8}
)

# Overlay a marker on the WT cells so it's clear they are the reference
for i, geno in enumerate(GENOTYPE_ORDER):
    if geno == 'WT':
        for j in range(pivot_rel.shape[1]):
            ax.text(j + 0.5, i + 0.72, '(ref)',
                    ha='center', va='center',
                    fontsize=7, color='#666666', style='italic')

ax.set_title('Cox2-sfGFP expression\nrelative to WT', fontsize=11, pad=8)
ax.set_xlabel('Temperature (°C)', fontsize=10)
ax.set_ylabel('')
ax.set_xticklabels(['30 °C', '37 °C'])
sns.despine(ax=ax, left=True, bottom=True)
plt.tight_layout()
plt.rcParams['svg.fonttype'] = 'none'
plt.savefig(f'{OUT_DIR}/plot_relative_090626_heatmap.svg')
plt.show()